# Asistente de anotación

Este notebook utiliza un modelo de lenguaje para proponer la función que cumple cada cita en un texto académico y justificar su elección. Guarda las propuestas en una copia del archivo para que una persona pueda revisarlas y corregirlas antes de utilizarlas en la evaluación de los modelos.


## 1. Preparar el entorno

Las funciones compartidas están en `anotacion_asistida.py`, en el mismo directorio que este notebook. Mantén ambos archivos juntos. Solo se necesita el paquete `openai` para las llamadas al modelo; si falta, ejecuta la celda de instalación opcional.

La clave se lee de `OPENAI_API_KEY` o se solicita de forma oculta al ejecutar. No la escribas en el notebook.


In [ ]:
from pathlib import Path
import importlib
import sys
from collections import Counter

# Funciona al abrir Jupyter desde la raíz o desde notebooks/.
candidates = [Path.cwd(), Path.cwd() / "notebooks"]
for parent in Path.cwd().parents:
    candidates.extend([parent, parent / "notebooks"])
MODULE_DIR = next((p for p in candidates if (p / "anotacion_asistida.py").is_file()), None)
if MODULE_DIR is None:
    raise FileNotFoundError("Coloca anotacion_asistida.py junto a este notebook.")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))
import anotacion_asistida
importlib.reload(anotacion_asistida)
from anotacion_asistida import CONFIG, PROJECT_ROOT, read_template, load_state, describe, run

print("Archivos disponibles:")
for file in sorted((PROJECT_ROOT / "tmp" / "Tanda 1").glob("*.xlsx")):
    print(" -", file.relative_to(PROJECT_ROOT))

## 2. Elegir archivo

Lo único que se configura es el Excel de entrada; por defecto, la hoja de Zambrana. Modelo (`gpt-4o-mini`), temperatura 0, reintentos y carpetas están reunidos en el diccionario `CONFIG` de `anotacion_asistida.py`. Las llamadas al proveedor pueden generar cargos.

In [ ]:
INPUT_FILE =  PROJECT_ROOT / "tmp/Tanda 1/TestGold_T1_v2_Arias.xlsx"
LOG_FILE = CONFIG["log_dir"] / INPUT_FILE.stem / "anotaciones.jsonl"

## 3. Comprobar el estado

La guía sale de `9_funciones` del mismo Excel. Solo se envían el contexto original en inglés y el título y resumen originales del citado; no se usan traducciones ni respuestas existentes. Las filas con cualquier respuesta en I–L se conservan, y los casos sin exactamente un `TARGETCIT` no se envían.

Esta celda no llama al modelo. Los casos que acumulan 6 respuestas inválidas con el método actual pasan a anotación humana. Los fallos del método anterior se conservan en el historial, pero permiten un nuevo intento con selección de evidencia.

In [ ]:
template = read_template(INPUT_FILE)
state = load_state(template, LOG_FILE)
print(describe(template, state))

## 4. Procesar pendientes y actualizar la salida

La siguiente celda hace lo mismo que `python notebooks/anotacion_asistida.py <archivo>`: solicita todos los pendientes y crea o actualiza `notebooks/anotaciones/<nombre>_output.xlsx`, junto con `_output.revision.csv` y `_output.procedencia.json`. Si no hay pendientes, solo actualiza la salida y no pide la clave.

- Cada intento se guarda al momento en `artifacts/anotacion_llm/<nombre>/anotaciones.jsonl`; si se interrumpe, vuelve a ejecutar y se reanuda.
- Un error de autenticación detiene el proceso; los errores transitorios y las respuestas inválidas se reintentan, y el reintento incluye la respuesta rechazada y el motivo.
- El modelo selecciona la evidencia entre fragmentos exactos del contexto que incluyen TARGETCIT; no necesita escribirla libremente. El esquema JSON limita las opciones y la validación sigue comprobando el texto original. Esto evita errores de copia, traducción o mezcla de frases, pero no garantiza que la categoría sea correcta. La confianza la declara el modelo y no está calibrada.
- Solo se rellenan I–L en filas vacías de la salida: tus correcciones se conservan, igual que las columnas `estado_revision`, `revisor`, `fecha` y `notas` del CSV. Cierra el Excel antes de ejecutar.
- No publiques la salida ni las respuestas individuales en GitHub.

In [ ]:
report = run(INPUT_FILE)

## 5. Resumen para revisión

Distribución de las propuestas guardadas. No se muestran respuestas de otros anotadores ni se marca nada como revisado automáticamente.

In [ ]:
state = load_state(template, LOG_FILE)
results = [record["result"] for record in state["proposals"].values()]
print(describe(template, state))
print("Evaluabilidad:", dict(Counter(r["evaluabilidad"] for r in results)))
print("Confianza declarada:", dict(Counter(r["confianza"] for r in results)))
print("Función propuesta:", dict(Counter(r["funcion"] for r in results)))
if report:
    print("Salida:", report["output"])
    print("Revisión:", report["review"])

## 6. Revisión humana y entrega

1. Abre `notebooks/anotaciones/<nombre>_output.xlsx` y revisa las decisiones. Usa el CSV auxiliar para priorizar ambigüedades, no evaluables, confianza baja, filas sin propuesta e intentos agotados. Revisa también casos restantes: dos respuestas coherentes pueden estar equivocadas.
2. Comprueba la categoría y la justificación contra el inglés original; corrige I–L directamente en la salida cuando corresponda. No cambies IDs ni textos originales, ni insertes o reordenes filas.
3. Registra en el CSV qué casos revisaste, quién lo hizo y cuándo. Esas columnas se conservan en las siguientes actualizaciones. El notebook nunca marca esa revisión por ti.
4. Conserva registro y procedencia junto a la entrega. Si el equipo acepta usarla, comparte la versión revisada por el canal acordado; declara que fue preanotada automáticamente y revisada después. No sustituye el requisito vigente de anotación humana ciega independiente.

Referencia técnica: [salidas estructuradas de OpenAI](https://developers.openai.com/api/docs/guides/structured-outputs?api-mode=chat).